# ⚡ TITAN-3 Protocol
### Structurally Validated Lightweight Secure Command Framework

**Team:** Vajra Protocol &nbsp;|&nbsp; Prakhar Dwivedi · Pramod Batham · Sumit Yadav  
**Institution:** Rustamji Institute of Technology, Gwalior  
**Research:** [DOI: 10.5281/zenodo.18814740](https://doi.org/10.5281/zenodo.18814740)  
**Submission:** India Innovation 2026

---

## How It Works

```
ENCRYPT                              DECRYPT
────────                             ────────
Any Command (string)                 Encrypted Packet
      │                                    │
      ▼                                    ▼
 Time → N (board size)             Same Clock → Same N
 Command Hash → Start Col          Same Command → Same Start
      │                                    │
      ▼                                    ▼
 Knight-Relaxed N-Queens           Knight-Relaxed N-Queens
 Finds ONE solution → Key          Same solution → Same Key
      │                            (NO key transmitted!)
      ▼                                    │
 Wrap in 3 random brackets         XOR decrypt → unwrap
 e.g. <<{COMMAND}>>                        │
      │                                    ▼
      ▼                            PDA checks brackets
 XOR encrypt → Base64              Tampered? → REJECT ✗
      │                            Valid?    → ACCEPT ✓
      ▼                                    │
 Encrypted Packet                  Original Command
```

**Run all cells top to bottom. Cell 2 loads the engine. Cells 3-6 are demos.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — TITAN-3 ENGINE (run this first)
# Pure Python standard library — zero dependencies
# ═══════════════════════════════════════════════════════════
import random, base64, time, statistics, hashlib
from datetime import datetime

DX = [2, 1, -1, -2, -2, -1, 1, 2]
DY = [1, 2,  2,  1, -1, -2, -2, -1]

def get_N_from_time():
    now = datetime.now()
    idx = (now.hour + now.minute) % 8
    return idx + 4, now.hour, now.minute

def command_to_params(command, N):
    h = hashlib.sha256(command.upper().encode()).digest()
    start_col = int.from_bytes(h[0:4], 'big') % N
    cols = list(range(N))
    for i in range(N-1, 0, -1):
        bi = (i*3) % 32
        j = int.from_bytes(h[bi:bi+2], 'big') % (i+1)
        cols[i], cols[j] = cols[j], cols[i]
    return start_col, cols

def hybrid_queens_keygen(N, start_col, col_priority):
    """Dwivedi's Knight-Relaxed N-Queens — DOI:10.5281/zenodo.18814740"""
    queens = [-1]*N; col_mask = diag1 = diag2 = 0
    def is_safe(r,c): return not((col_mask>>c)&1 or (diag1>>(r-c+N))&1 or (diag2>>(r+c))&1)
    def place(r,c):
        nonlocal col_mask,diag1,diag2
        queens[r]=c; col_mask|=(1<<c); diag1|=(1<<(r-c+N)); diag2|=(1<<(r+c))
    def unplace(r,c):
        nonlocal col_mask,diag1,diag2
        queens[r]=-1; col_mask^=(1<<c); diag1^=(1<<(r-c+N)); diag2^=(1<<(r+c))
    def solve(row, pr, pc):
        if row==N: return True
        tried=False
        for d in range(8):
            kr,kc=pr+DX[d],pc+DY[d]
            if kr==row and 0<=kc<N and is_safe(row,kc):
                tried=True; place(row,kc)
                if solve(row+1,row,kc): return True
                unplace(row,kc)
        if not tried:
            for col in col_priority:
                if is_safe(row,col):
                    place(row,col)
                    if solve(row+1,row,col): return True
                    unplace(row,col)
        return False
    place(0,start_col)
    if solve(1,0,start_col): return list(queens)
    unplace(0,start_col); return None

def get_key(command):
    N,h,m = get_N_from_time()
    sc,cp = command_to_params(command, N)
    sol = None
    for off in range(N):
        s = hybrid_queens_keygen(N,(sc+off)%N,cp)
        if s: sol=s; break
    if not sol: return 'FAILSAFE0000000000000000000000000', N, h, m, None
    raw = ''.join(str(x) for x in sol)
    ch  = hashlib.sha256(command.upper().encode()).hexdigest()
    key = ''.join(str((int(raw[i%len(raw)])+int(ch[i%len(ch)],16))%10) for i in range(32))
    return key, N, h, m, sol

class XOR_Engine:
    def __init__(self,key): self.key=key
    def process(self,data):
        r=''.join(chr(ord(c)^ord(self.key[i%len(self.key)])) for i,c in enumerate(data))
        return base64.b64encode(r.encode('latin-1')).decode()
    def deprocess(self,b64):
        raw=base64.b64decode(b64).decode('latin-1')
        return ''.join(chr(ord(c)^ord(self.key[i%len(self.key)])) for i,c in enumerate(raw))

class GrammarWrapper:
    P={'[':']','{':'}','<':'>','(':')'}; O=list(P.keys())
    def wrap(self,cmd):
        L=random.choices(self.O,k=3); p=cmd
        for o in reversed(L): p=f'{o}{p}{self.P[o]}'
        return p
    def unwrap(self,p):
        while p and p[0] in self.O and p[-1]==self.P[p[0]]: p=p[1:-1]
        return p

class PDA:
    OP={'[','{','<','('}; CL={']','}','>',')'}
    M={']':'[','}':'{','>':'<',')':'('}
    def validate(self,pkt):
        stack=['$']
        for ch in pkt:
            if ch in self.OP: stack.append(ch)
            elif ch in self.CL:
                if len(stack)==1: return False,'✗ Stack underflow — TAMPERED'
                if stack.pop()!=self.M[ch]: return False,'✗ Bracket mismatch — TAMPERED'
        return (stack==['$']),('✓ Integrity confirmed' if stack==['$'] else '✗ Unclosed brackets — TAMPERED')

class Titan3:
    def __init__(self): self.W=GrammarWrapper(); self.pda=PDA()
    def encrypt(self,cmd):
        key,N,h,m,sol=get_key(cmd)
        wrapped=self.W.wrap(cmd)
        packet=XOR_Engine(key).process(wrapped)
        return packet,key,N,h,m,sol,wrapped
    def decrypt(self,pkt,cmd):
        key,N,h,m,sol=get_key(cmd)
        unwrapped=XOR_Engine(key).deprocess(pkt)
        valid,reason=self.pda.validate(unwrapped)
        return (self.W.unwrap(unwrapped) if valid else None),valid,reason,unwrapped

titan = Titan3()
print('✅ TITAN-3 Engine loaded — zero dependencies, pure Python!')
print(f'🕐 Current time derived N = {get_N_from_time()[0]} (board size changes every session)')

## 🔐 Demo 1 — Full Encrypt → Decrypt Pipeline

In [ ]:
print('='*65)
print('  DEMO 1: FULL ENCRYPT → DECRYPT PIPELINE')
print('='*65)

commands = ['MOVE', 'FIRE_DRONE', 'SCAN_BOOTH_01', 'HALT_UNIT_X9', 'STATUS_NODE_N1']

for cmd in commands:
    print(f'\n  Command  : {cmd}')
    pkt,key,N,h,m,sol,wrapped = titan.encrypt(cmd)
    print(f'  Time     : {h:02d}:{m:02d} → Board size N={N}')
    print(f'  Solution : {sol}')
    print(f'  Key      : {key}')
    print(f'  Wrapped  : {wrapped}')
    print(f'  Packet   : {pkt}')
    recovered,valid,reason,_ = titan.decrypt(pkt,cmd)
    print(f'  PDA      : {reason}')
    print(f'  Recovered: {recovered}  {"✓ MATCH" if recovered==cmd else "✗ FAIL"}')

## 🛡️ Demo 2 — Tamper Detection

In [ ]:
print('='*65)
print('  DEMO 2: TAMPER DETECTION')
print('='*65)
print('  If even one byte of the encrypted packet is changed,')
print('  the PDA detects structural corruption and REJECTS it.\n')

for cmd in ['HALT', 'MOVE_DRONE', 'SCAN_BOOTH_01']:
    pkt,*_ = titan.encrypt(cmd)
    corrupted = pkt[:-4] + 'XXXX'
    _,valid,reason,_ = titan.decrypt(corrupted,cmd)
    print(f'  Command  : {cmd}')
    print(f'  Original : {pkt}')
    print(f'  Tampered : {corrupted}')
    print(f'  PDA      : {reason}')
    print(f'  Blocked  : {not valid} ✓\n')

## 📊 Demo 3 — Benchmark

In [ ]:
print('='*65)
print('  DEMO 3: PERFORMANCE BENCHMARK — 500 commands')
print('='*65)

pool = ['MOVE','FIRE','SCAN','HALT','STATUS']
times_enc, times_pda = [], []
pda_only = PDA()
wrapper  = GrammarWrapper()

for _ in range(30):  # warm-up
    cmd=random.choice(pool); pkt,*_=titan.encrypt(cmd); titan.decrypt(pkt,cmd)

for _ in range(500):
    cmd=random.choice(pool)
    t0=time.perf_counter(); pkt,*_=titan.encrypt(cmd); times_enc.append(time.perf_counter()-t0)
    tp0=time.perf_counter(); pda_only.validate(wrapper.wrap(cmd)); times_pda.append(time.perf_counter()-tp0)

print(f'\n  Commands Tested      : 500')
print(f'  Avg Encrypt Latency  : {statistics.mean(times_enc)*1000:.4f} ms')
print(f'  PDA Validation Only  : {statistics.mean(times_pda)*1000:.4f} ms  ← structural check alone')
print(f'  Throughput           : {500/sum(times_enc):.0f} ops/sec')
print(f'  Std Deviation        : {statistics.stdev(times_enc)*1000:.4f} ms')
print(f'  Min Latency          : {min(times_enc)*1000:.4f} ms')
print(f'  Zero external deps   : ✓ (pure Python stdlib)')
print(f'  Key transmitted      : ✗ None (derived from command + time)')

## ✏️ Demo 4 — Try Any Command Yourself

In [ ]:
# ✏️ CHANGE THIS TO ANYTHING YOU WANT AND RUN THE CELL
my_command = 'VAJRA_PROTOCOL_2026'

print('='*65)
print(f'  YOUR COMMAND: {my_command}')
print('='*65)

pkt,key,N,h,m,sol,wrapped = titan.encrypt(my_command)
print(f'  Board size N : {N}  (from {h:02d}:{m:02d})')
print(f'  Solution     : {sol}')
print(f'  Key          : {key}')
print(f'  Wrapped      : {wrapped}')
print(f'  Encrypted    : {pkt}')
print()
recovered,valid,reason,_ = titan.decrypt(pkt,my_command)
print(f'  PDA Check    : {reason}')
print(f'  Recovered    : {recovered}')
print(f'  Match        : {recovered == my_command} ✓')
print()
print('  Now change my_command to anything and run again!')